<a href="https://colab.research.google.com/github/srkprattipati/Bank-marketing_ML_assignment_2/blob/project/ML_assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
import joblib

# Load dataset
df = pd.read_csv("/content/bank-full.csv", sep=";")

# Encode target
label_encoder = LabelEncoder()
df["y"] = label_encoder.fit_transform(df["y"])

# Features & target
X = df.drop("y", axis=1)
y = df["y"]

# One-hot encode
X = pd.get_dummies(X, drop_first=True)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Save test data
test_df = X_test.copy()
test_df["y"] = y_test
test_df.to_csv("test_data.csv", index=False)

# Create model folder
!mkdir -p model

# Models
models = {
    "Logistic Regression":LogisticRegression(
    max_iter=5000,
    solver="liblinear",
    class_weight="balanced"
),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "kNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

results = []

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)

    # Save model
    joblib.dump(model, f"model/{name.replace(' ', '_').lower()}.joblib")

    # Predictions
    y_pred = model.predict(X_test)

    # AUC
    try:
        y_proba = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_proba)
    except:
        auc = None

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": acc,
        "AUC": auc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "MCC": mcc,
    })

    print(classification_report(y_test, y_pred))
    print("-" * 60)

# Save metrics
results_df = pd.DataFrame(results)
results_df.to_csv("model_metrics.csv", index=False)

print("Training complete. Models saved in model/")


FileNotFoundError: [Errno 2] No such file or directory: '/content/bank-full.csv'